# 09d — Feedforward Network on Tabular Data (Wine Dataset)

Corresponds to: Note 09 — Deep Feedforward Network Architectures

**Libraries introduced in this notebook:** `sklearn.datasets.load_wine` — provides the UCI Wine dataset (13 chemical features, 3 wine cultivar classes, 178 samples); `sklearn.preprocessing.StandardScaler` — standardises features to zero mean and unit variance.

**What this notebook demonstrates:**
1. Loading and inspecting the Wine dataset — 13 features, 3 classes, 178 samples
2. Preprocessing — train/test split, feature standardisation
3. Building and training a feedforward network on tabular data
4. Evaluating accuracy and examining per-class confidence
5. Comparing network depth — shallow vs deep on a small tabular dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
print(f'PyTorch: {torch.__version__}')

## Part 1: Load and Inspect the Wine Dataset

178 samples of wine from three Italian cultivars. 13 chemical measurements per sample (alcohol, malic acid, ash, etc.). Task: classify which cultivar.

In [ ]:
data = load_wine()
X, y = data.data, data.target

print(f'Samples:  {X.shape[0]}')
print(f'Features: {X.shape[1]}')
print(f'Classes:  {list(data.target_names)}')
print(f'Class distribution: {[(y == c).sum() for c in range(3)]}')
print()
print('Features (name, range):')
for i, name in enumerate(data.feature_names):
    print(f'  {i+1:2}. {name:<35}  [{X[:,i].min():.2f}, {X[:,i].max():.2f}]')

## Part 2: Preprocess

Features have very different scales (alcohol ~12, proline ~700). Standardise to zero mean and unit variance using training statistics only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.long)

print(f'Train: {X_tr.shape}  Test: {X_te.shape}')
print(f'Train class distribution: {[(y_tr == c).sum().item() for c in range(3)]}')
print(f'Test  class distribution: {[(y_te == c).sum().item() for c in range(3)]}')

## Part 3: Build and Train

In [ ]:
model = nn.Sequential(
    nn.Linear(13, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32,  3),
)

print(f'Architecture: [13, 64, 32, 3]')
print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print()
print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<14} {"Test Loss":<14} {"Test Acc"}')
print('-' * 60)

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    out  = model(X_tr)
    loss = criterion(out, y_tr)
    loss.backward()
    optimizer.step()
    tr_acc = (out.argmax(1) == y_tr).float().mean().item()

    model.eval()
    with torch.no_grad():
        out_te  = model(X_te)
        te_loss = criterion(out_te, y_te).item()
        te_acc  = (out_te.argmax(1) == y_te).float().mean().item()

    if epoch % 50 == 0 or epoch == 1:
        print(f'{epoch:<8} {loss.item():<14.4f} {tr_acc:<14.4f} {te_loss:<14.4f} {te_acc:.4f}')

## Part 4: Per-Class Confidence

In [ ]:
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(X_te), dim=1)
    preds = probs.argmax(1)

print('Test set predictions (first 10):')
print(f'{"Sample":<8} {"True":<8} {"Pred":<8} {"P(0)":<10} {"P(1)":<10} {"P(2)":<10} {"Correct"}')
print('-' * 62)
for i in range(10):
    p = probs[i].numpy()
    correct = 'yes' if preds[i] == y_te[i] else 'NO'
    print(f'{i:<8} {y_te[i].item():<8} {preds[i].item():<8} {p[0]:<10.3f} {p[1]:<10.3f} {p[2]:<10.3f} {correct}')

print()
print(f'Overall test accuracy: {(preds == y_te).float().mean().item():.4f}')

## Part 5: Depth Comparison

In [ ]:
def build_and_train(layers, epochs=200):
    modules = []
    for i in range(len(layers) - 1):
        modules.append(nn.Linear(layers[i], layers[i+1]))
        if i < len(layers) - 2:
            modules.append(nn.ReLU())
    net = nn.Sequential(*modules)
    opt = optim.Adam(net.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        crit(net(X_tr), y_tr).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        te_acc = (net(X_te).argmax(1) == y_te).float().mean().item()
        params = sum(p.numel() for p in net.parameters())
    return te_acc, params

configs = [
    ([13, 64, 3],          'Shallow  [13, 64, 3]'),
    ([13, 64, 32, 3],      'Medium   [13, 64, 32, 3]'),
    ([13, 64, 32, 16, 3],  'Deep     [13, 64, 32, 16, 3]'),
]

print('Depth comparison (200 epochs each):')
print(f'{"Architecture":<30} {"Parameters":<14} {"Test Accuracy"}')
print('-' * 56)
for layers, label in configs:
    torch.manual_seed(42)
    acc, params = build_and_train(layers)
    print(f'{label:<30} {params:<14} {acc:.4f}')

## Summary

What this notebook demonstrated:

1. **Wine dataset** — 178 samples, 13 chemical features, 3 cultivar classes. A real tabular dataset — the natural domain for feedforward networks.
2. **Preprocessing** — feature standardisation is essential when features have different scales. Scaler fit on training data only.
3. **Training** — 142 training samples fit in one batch. Cross-entropy loss, Adam lr=1e-3, 200 epochs.
4. **Per-class confidence** — softmax probabilities show how certain the network is. High confidence on correct predictions.
5. **Depth comparison** — on small tabular datasets, adding more layers does not always improve accuracy.